# Distribución transversal mediante modelo de parrilla

Este notebook ejecuta el análisis paso a paso sin depender de la interfaz de terminal. El solver permanece en el paquete `analisis_superestructura`, por lo que la formulación, las validaciones y la configuración YAML siguen siendo portátiles y reutilizables.

La parrilla usa tres grados de libertad por nodo ($w$, $\theta_x$, $\theta_y$), vigas longitudinales con rigidez compuesta de corto plazo y franjas transversales de losa. Los resultados son factores globales para las vigas; no corresponden a esfuerzos locales de losa ni al diseño de diafragmas.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

def encontrar_raiz(inicio: Path) -> Path:
    inicio = inicio.resolve()
    for candidato in (inicio, *inicio.parents):
        if (candidato / 'pyproject.toml').exists() and (candidato / 'analisis_superestructura').exists():
            return candidato
    raise FileNotFoundError('No se encontró la raíz del proyecto')

RAIZ = encontrar_raiz(Path.cwd())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from analisis_superestructura.vigas_principales import (
    analizar,
    analizar_distribucion_parrilla,
    validar_configuracion,
)
from analisis_superestructura.vigas_principales.analisis.distribucion import calcular_factores_distribucion
from analisis_superestructura.vigas_principales.analisis.secciones import propiedades_acero
from analisis_superestructura.vigas_principales.reportes.exportar import exportar_analisis

print(f'Raíz del proyecto: {RAIZ}')

## 1. Opciones de ejecución

Edite únicamente esta celda para cambiar el caso, la malla o el barrido. El YAML conserva todos los datos estructurales; las opciones siguientes son sobreescrituras temporales y trazables del notebook.

In [ ]:
CONFIG_YAML = RAIZ / 'analisis_superestructura/vigas_principales/ejemplos/molinohuayco_preliminar.yaml'

OPCIONES_PARRILLA = {
    'numero_tramos_longitudinales': 20,
    'paso_posicion_transversal': 0.10,       # m
    'paso_busqueda_longitudinal': 0.25,      # m
    'ancho_carril_diseno': 3.00,             # m
    'separacion_lineas_rueda': 1.80,         # m
    'factor_presencia_multiple': 1.20,
    'factor_rigidez_flexion_transversal': 1.0,
    'factor_rigidez_torsional': 1.0,
    'puntos_integracion_carga_carril': 5,
}

EJECUTAR_CONVERGENCIA = True
MALLAS_CONVERGENCIA = (10, 20, 40)
EXPORTAR_RESULTADOS = False
DIRECTORIO_SALIDA = RAIZ / 'outputs/vigas_principales'

## 2. Carga y validación de datos

In [ ]:
config_base = validar_configuracion(CONFIG_YAML)
parrilla = config_base.analisis.parrilla.model_copy(update=OPCIONES_PARRILLA)
analisis_cfg = config_base.analisis.model_copy(
    update={'metodo_distribucion': 'parrilla', 'parrilla': parrilla}
)
config = config_base.model_copy(update={'analisis': analisis_cfg})

g = config.geometria
display(Markdown(
    f'**Proyecto:** {config.proyecto.nombre}  \n'
    f'**Luz:** {g.luz:.2f} m · **Vigas:** {g.numero_vigas} · '
    f'**Separación:** {g.separacion_vigas:.2f} m · **Tablero:** {g.ancho_tablero:.2f} m  \n'
    f'**Método activo:** `{config.analisis.metodo_distribucion}`'
))
display(config.analisis.parrilla.model_dump())

## 3. Ejecución del modelo refinado

In [ ]:
resultado_parrilla = analizar_distribucion_parrilla(config)
f = resultado_parrilla.factores

filas = [
    ('Momento interior', f.momento_interior),
    ('Momento exterior', f.momento_exterior),
    ('Corte interior', f.corte_interior),
    ('Corte exterior', f.corte_exterior),
    ('Fatiga interior', f.fatiga_interior),
    ('Fatiga exterior', f.fatiga_exterior),
]
tabla = ['| Factor | Parrilla |', '|---|---:|'] + [f'| {n} | {v:.6f} |' for n, v in filas]
display(Markdown('\n'.join(tabla)))
display(Markdown(
    f'**Grados de libertad:** {resultado_parrilla.numero_grados_libertad}  \n'
    f'**Error máximo de equilibrio:** {resultado_parrilla.error_equilibrio_maximo:.3e}'
))
for advertencia in resultado_parrilla.advertencias:
    display(Markdown(f'- {advertencia}'))

## 4. Distribución entre vigas según la posición transversal

Las curvas siguientes no incluyen el factor de presencia múltiple; muestran la fracción mecánica que recibe cada viga para cada posición del carril. La suma de cada fila debe ser uno.

In [ ]:
centros_m = resultado_parrilla.centros_carril_mm / 1000.0
fig, ejes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for i in range(config.geometria.numero_vigas):
    ejes[0].plot(centros_m, resultado_parrilla.participacion_momento[:, i], marker='o', label=f'Viga {i + 1}')
    ejes[1].plot(centros_m, resultado_parrilla.participacion_corte[:, i], marker='o', label=f'Viga {i + 1}')
for ax, titulo in zip(ejes, ('Momento en centro de luz', 'Reacción/corte en apoyo')):
    ax.set_title(titulo)
    ax.set_xlabel('Centro del carril respecto al eje del tablero (m)')
    ax.set_ylabel('Fracción del efecto longitudinal')
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.tight_layout()
plt.show()

assert np.allclose(resultado_parrilla.participacion_momento.sum(axis=1), 1.0, atol=1e-8)
assert np.allclose(resultado_parrilla.participacion_corte.sum(axis=1), 1.0, atol=1e-8)

## 5. Comparación con expresiones aproximadas

In [ ]:
representativo = propiedades_acero(config.viga.segmentos[len(config.viga.segmentos) // 2])
aproximados = calcular_factores_distribucion(config, representativo)
nombres = (
    'momento_interior', 'momento_exterior', 'corte_interior',
    'corte_exterior', 'fatiga_interior', 'fatiga_exterior',
)
tabla = ['| Factor | Aproximado MTC | Parrilla | Diferencia |', '|---|---:|---:|---:|']
for nombre in nombres:
    a = getattr(aproximados, nombre)
    p = getattr(f, nombre)
    tabla.append(f'| {nombre} | {a:.6f} | {p:.6f} | {(p / a - 1.0) * 100:+.2f} % |')
display(Markdown('\n'.join(tabla)))

## 6. Estudio opcional de convergencia de malla

In [ ]:
resultados_convergencia = []
if EJECUTAR_CONVERGENCIA:
    for n_tramos in MALLAS_CONVERGENCIA:
        p_cfg = parrilla.model_copy(update={'numero_tramos_longitudinales': n_tramos})
        a_cfg = analisis_cfg.model_copy(update={'parrilla': p_cfg})
        c_cfg = config.model_copy(update={'analisis': a_cfg})
        r = analizar_distribucion_parrilla(c_cfg)
        resultados_convergencia.append((
            n_tramos, r.factores.momento_interior, r.factores.momento_exterior,
            r.factores.corte_interior, r.factores.corte_exterior,
            r.error_equilibrio_maximo,
        ))
    tabla = ['| Tramos base | $g_{m,int}$ | $g_{m,ext}$ | $g_{v,int}$ | $g_{v,ext}$ | Error equilibrio |',
             '|---:|---:|---:|---:|---:|---:|']
    tabla += [f'| {n} | {gmi:.6f} | {gme:.6f} | {gvi:.6f} | {gve:.6f} | {err:.2e} |'
              for n, gmi, gme, gvi, gve, err in resultados_convergencia]
    display(Markdown('\n'.join(tabla)))
else:
    display(Markdown('*Estudio de convergencia desactivado.*'))

## 7. Integración con el análisis longitudinal completo

Esta celda confirma que el motor de vigas principales usa los factores refinados para momentos, cortantes, fatiga y deformaciones.

In [ ]:
resultado_analisis = analizar(config)
assert resultado_analisis.detalle_distribucion is not None
assert np.isclose(resultado_analisis.factores_distribucion['momento_interior'], f.momento_interior)

filas = []
for tipo in ('interior', 'exterior'):
    datos = resultado_analisis.tipos_viga[tipo]
    filas.append((
        tipo,
        np.max(datos['momentos_nmm']['LL_IM']) / 1e6,
        np.max(np.abs(datos['cortantes_n']['LL_IM_positivo'])) / 1000.0,
        np.max(np.abs(datos['deflexiones_mm']['LL_IM_critica'])),
    ))
tabla = [r'| Viga | $M_{LL+IM,max}$ (kN·m) | $V_{LL+IM,max}$ (kN) | $|\Delta_{LL}|_{max}$ (mm) |',
         '|---|---:|---:|---:|']
tabla += [f'| {tipo} | {mom:.2f} | {cor:.2f} | {deform:.2f} |' for tipo, mom, cor, deform in filas]
display(Markdown('\n'.join(tabla)))

## 8. Exportación opcional

Al activar `EXPORTAR_RESULTADOS`, el JSON de análisis incluye `detalle_distribucion` con malla, posiciones de carril, participaciones, casos críticos y control de equilibrio.

In [ ]:
if EXPORTAR_RESULTADOS:
    salida = exportar_analisis(resultado_analisis, DIRECTORIO_SALIDA)
    display(Markdown(f'Resultados exportados en: `{salida}`'))
else:
    display(Markdown('*Exportación desactivada; no se escribieron archivos.*'))